In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM ecommerce.base.customers
UNION ALL SELECT 'orders', COUNT(*) FROM ecommerce.base.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM ecommerce.base.order_items
UNION ALL SELECT 'products', COUNT(*) FROM ecommerce.base.products
UNION ALL SELECT 'payments', COUNT(*) FROM ecommerce.base.payments
UNION ALL SELECT 'returns', COUNT(*) FROM ecommerce.base.returns
UNION ALL SELECT 'regions', COUNT(*) FROM ecommerce.base.regions
UNION ALL SELECT 'stores', COUNT(*) FROM ecommerce.base.stores
UNION ALL SELECT 'categories', COUNT(*) FROM ecommerce.base.categories;

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT order_id, COUNT(*) 
FROM ecommerce.base.orders 
GROUP BY order_id 
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT category_id, COUNT(*) 
FROM ecommerce.base.categories 
GROUP BY category_id 
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT customer_id, COUNT(*) 
FROM ecommerce.base.customers 
GROUP BY customer_id 
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT product_id, COUNT(*) 
FROM ecommerce.base.products 
GROUP BY product_id 
HAVING COUNT(*) > 1; 

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT payment_id, COUNT(*) 
FROM ecommerce.base.payments 
GROUP BY payment_id 
HAVING COUNT(*) > 1; 

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT region_id, COUNT(*) 
FROM ecommerce.base.regions 
GROUP BY region_id 
HAVING COUNT(*) > 1; 

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT return_id, COUNT(*) 
FROM ecommerce.base.returns 
GROUP BY return_id 
HAVING COUNT(*) > 1; 

In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT store_id, COUNT(*) 
FROM ecommerce.base.stores 
GROUP BY store_id 
HAVING COUNT(*) > 1; 


In [0]:
%sql
-- Orphan check: orders referencing a customer_id that doesn't exist in customers
SELECT o.customer_id
FROM ecommerce.base.orders o
LEFT JOIN ecommerce.base.customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

In [0]:
%sql
-- Orphan check: orders referencing a region_id that doesn't exist in regions
SELECT o.region_id
FROM ecommerce.base.orders o
LEFT JOIN ecommerce.base.regions r ON o.region_id = r.region_id
WHERE r.region_id IS NULL;

In [0]:
%sql
-- Orphan check: orders referencing a store_id that doesn't exist in stores
SELECT o.store_id
FROM ecommerce.base.orders o
LEFT JOIN ecommerce.base.stores s ON o.store_id = s.store_id
WHERE s.store_id IS NULL;

In [0]:
%sql
-- Orphan check: products referencing a category_id that doesn't exist in categories
SELECT p.category_id
FROM ecommerce.base.products p
LEFT JOIN ecommerce.base.categories c ON p.category_id = c.category_id
WHERE c.category_id IS NULL;


In [0]:
%sql

-- Orphan check: payments referencing an order_id that doesn't exist in orders
SELECT p.order_id
FROM ecommerce.base.payments p
LEFT JOIN ecommerce.base.orders o ON p.order_id = o.order_id
WHERE o.order_id IS NULL;


In [0]:
%sql

-- Orphan check: returns referencing an order_id that doesn't exist in orders
SELECT r.order_id
FROM ecommerce.base.returns r
LEFT JOIN ecommerce.base.orders o ON r.order_id = o.order_id
WHERE o.order_id IS NULL;

In [0]:
%sql
-- Cardinality check: unique counts across key dimension columns
SELECT
  (SELECT COUNT(DISTINCT customer_id) FROM ecommerce.base.customers) AS unique_customers,
  (SELECT COUNT(DISTINCT product_id) FROM ecommerce.base.products) AS unique_products,
  (SELECT COUNT(DISTINCT region_id) FROM ecommerce.base.regions) AS unique_regions,
  (SELECT COUNT(DISTINCT store_id) FROM ecommerce.base.stores) AS unique_stores,
  (SELECT COUNT(DISTINCT category_id) FROM ecommerce.base.categories) AS unique_categories,
  (SELECT COUNT(DISTINCT brand) FROM ecommerce.base.products) AS unique_brands,
  (SELECT COUNT(DISTINCT city) FROM ecommerce.base.customers) AS unique_cities;

In [0]:
%sql
-- Full-row duplicate check: same customer, same order date, same total (likely double-entry)
SELECT customer_id, order_date, order_total, COUNT(*) AS occurrences
FROM ecommerce.base.orders
GROUP BY customer_id, order_date, order_total
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Full-row duplicate check: same order, same product, same quantity/price repeated
SELECT order_id, product_id, quantity, unit_price, COUNT(*) AS occurrences
FROM ecommerce.base.order_items
GROUP BY order_id, product_id, quantity, unit_price
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT order_id, product_id, quantity, unit_price,
       COUNT(*) AS occurrences,
       COUNT(DISTINCT order_item_id) AS unique_items
FROM ecommerce.base.order_items
GROUP BY order_id, product_id, quantity, unit_price
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    first_name,
    last_name,
    city,
    birth_date,
    COUNT(*) AS occurrences
FROM ecommerce.base.customers
GROUP BY first_name, last_name, city, birth_date
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

In [0]:
%sql
SELECT
    city,
    COUNT(*) AS occurrences
FROM ecommerce.base.customers
WHERE LOWER(TRIM(city)) IN ('n/a', 'unknown', '-')
GROUP BY city
ORDER BY occurrences DESC;